# AutoData Agent Colab Runner

This notebook runs the AutoData research pipeline from the plan:

`evaluate -> diagnose -> plan -> generate -> verify -> mix -> fine-tune -> re-evaluate`

Start with `RUN_MODE = "smoke"`. After that succeeds, switch to `prototype` with real MedMCQA and Qwen evaluation/training.

## 1. No Google Drive Required

This notebook saves outputs to the Colab runtime by default, so it does not mount or require Google Drive.

In [ ]:
from pathlib import Path

LOCAL_OUTPUT_DIR = Path('/content/autodata_outputs')
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Outputs will be saved under:', LOCAL_OUTPUT_DIR)

## 2. Locate Or Clone The Repository

If you opened this notebook directly in Colab, it clones the repository into `/content/autodata-agent`.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_URL = 'https://github.com/Jiaqi-Ye/AutoData.git'
REPO_DIR = '/content/autodata-agent'

current = Path.cwd()
if (current / 'autodata').exists() and (current / 'configs').exists():
    repo_path = current
else:
    repo_path = Path(REPO_DIR)
    if not (repo_path / 'autodata').exists():
        if not REPO_URL.strip():
            raise RuntimeError(
                'Set REPO_URL to your autodata-agent GitHub repo.'
            )
        if repo_path.exists():
            shutil.rmtree(repo_path)
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(repo_path)], check=True)

os.chdir(repo_path)
if Path.cwd().as_posix() == '/content/autodata-agent' and (Path.cwd() / '.git').exists():
    subprocess.run(['git', 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', 'reset', '--hard', 'origin/main'], check=True)
    subprocess.run(['git', 'log', '-1', '--oneline'], check=True)
print('Repository:', Path.cwd())
print('Files:', sorted(p.name for p in Path.cwd().iterdir())[:10])

## 3. Install Dependencies

Smoke mode installs only lightweight packages. Prototype/full mode needs the ML stack. On Colab GPU, `bitsandbytes` is supported.

In [ ]:
INSTALL_DEPS = True

LIGHTWEIGHT_DEPS = [
    'pyyaml>=6.0',
    'pytest>=7.0',
    'tqdm>=4.66',
]

REAL_DEPS = [
    'pyyaml>=6.0',
    'pytest>=7.0',
    'tqdm>=4.66',
    'datasets>=2.18',
    'torch>=2.1',
    'transformers>=4.43',
    'accelerate>=0.31',
    'peft>=0.11',
    'trl>=0.9',
    'bitsandbytes>=0.43',
]

OPENAI_DEPS = [
    'openai>=1.0',
]

if INSTALL_DEPS:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *LIGHTWEIGHT_DEPS])
print('Dependency step complete.')

## 4. Choose Run Settings

This notebook is configured for the GPT synthetic scaling experiment:

- `RUN_SYNTHETIC_SCALING = True`
- `SYNTHETIC_SCALE_BUDGETS = [500, 1000]`
- GPT generation through `GENERATION_PROVIDER = 'openai'`
- OpenAI LLM planning through `PLANNING_STRATEGY = 'llm_agent'`
- medical critic disabled for this scaling run
- QLoRA training enabled so each budget produces an after-training accuracy point


In [ ]:
RUN_MODE = "prototype"
RUN_STRATEGY_COMPARISON = False
RUN_SYNTHETIC_SCALING = True
SYNTHETIC_SCALE_BUDGETS = [500, 1000]

USE_REAL_MODEL = True
USE_REAL_TRAINING = True
USE_MOCK_GENERATION = False
GENERATION_PROVIDER = "openai"
GENERATION_MODEL = "Qwen/Qwen2.5-7B-Instruct"
GENERATION_API_MODEL = "gpt-4o-mini"
GENERATION_API_BATCH_SIZE = 5
OPENAI_API_KEY_ENV = "OPENAI_API_KEY"

PLANNING_STRATEGY = "llm_agent"
LLM_AGENT_PROVIDER = "openai"
LLM_AGENT_MODEL = "gpt-4o-mini"

USE_MEDICAL_CRITIC = False
MEDICAL_CRITIC_PROVIDER = "openai"
MEDICAL_CRITIC_MODEL = "gpt-4o-mini"
MEDICAL_CRITIC_LOCAL_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
MEDICAL_CRITIC_API_KEY_ENV = OPENAI_API_KEY_ENV
MEDICAL_CRITIC_FAIL_CLOSED = True

OUTPUT_DIR = "/content/autodata_outputs"

EVAL_SAMPLES_PER_DOMAIN = 20
TRAIN_POOL_SAMPLES_PER_DOMAIN = 100
TOTAL_SYNTHETIC_BUDGET = SYNTHETIC_SCALE_BUDGETS[0]
MIN_SAMPLES_PER_DOMAIN = 25
MAX_TRAINING_STEPS = 50

print("Run mode:", RUN_MODE)
print("Strategy comparison:", RUN_STRATEGY_COMPARISON)
print("Synthetic scaling:", RUN_SYNTHETIC_SCALING, SYNTHETIC_SCALE_BUDGETS)
print("Planning strategy:", PLANNING_STRATEGY, LLM_AGENT_PROVIDER)
print("Generation provider:", GENERATION_PROVIDER)
print("Generation model:", GENERATION_API_MODEL if GENERATION_PROVIDER == "openai" else GENERATION_MODEL)
print("Medical critic:", USE_MEDICAL_CRITIC, MEDICAL_CRITIC_PROVIDER)
print("Real training:", USE_REAL_TRAINING)


In [ ]:
import os

api_key_env = globals().get("OPENAI_API_KEY_ENV", "OPENAI_API_KEY")
raw_key = os.environ.get(api_key_env, "")

if not raw_key:
    try:
        from google.colab import userdata

        raw_key = userdata.get(api_key_env) or ""
    except Exception:
        raw_key = ""

clean_key = "".join(str(raw_key).split())
if clean_key:
    os.environ[api_key_env] = clean_key

print(f"{api_key_env} loaded:", bool(os.environ.get(api_key_env)))
print("length:", len(os.environ.get(api_key_env, "")))
print("has whitespace:", any(ch.isspace() for ch in os.environ.get(api_key_env, "")))


In [ ]:
import os, urllib.request, urllib.error

api_key_env = globals().get("OPENAI_API_KEY_ENV", "OPENAI_API_KEY")
key = os.environ.get(api_key_env, "")
if not key:
    raise RuntimeError(f"Set {api_key_env} in Colab Secrets before using GPT generation or the LLM planner.")

req = urllib.request.Request(
    "https://api.openai.com/v1/models",
    headers={"Authorization": f"Bearer {key}"},
)

try:
    with urllib.request.urlopen(req, timeout=30) as resp:
        print("OpenAI reachable:", resp.status)
except urllib.error.HTTPError as e:
    print("HTTP status:", e.code)
    print(e.read().decode("utf-8")[:1000])
    raise
except Exception as e:
    print(type(e).__name__, str(e))
    raise


## 5. Install Real-Run Dependencies If Needed

This cell only installs the larger ML stack when your settings require it.

In [ ]:
needs_medical_critic_stack = USE_MEDICAL_CRITIC and MEDICAL_CRITIC_PROVIDER in {"local_hf", "qwen", "local_qwen"}
needs_llm_agent_stack = PLANNING_STRATEGY == "llm_agent" and LLM_AGENT_PROVIDER == "openai"
needs_openai_stack = (GENERATION_PROVIDER == "openai") or (USE_MEDICAL_CRITIC and MEDICAL_CRITIC_PROVIDER == "openai") or needs_llm_agent_stack
needs_real_stack = USE_REAL_MODEL or USE_REAL_TRAINING or (not USE_MOCK_GENERATION) or needs_medical_critic_stack
if INSTALL_DEPS and needs_real_stack:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *REAL_DEPS])
if INSTALL_DEPS and needs_openai_stack:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *OPENAI_DEPS])
print('Real-stack needed:', needs_real_stack)
print('OpenAI client needed:', needs_openai_stack)


## 6. Build Runtime Config

This creates a temporary config from `configs/{RUN_MODE}_colab.yaml` and applies the notebook controls above.

In [ ]:
import os
import yaml
from pathlib import Path

from autodata.config import save_config

config_path = Path("configs") / f"{RUN_MODE}_colab.yaml"
if not config_path.exists():
    raise FileNotFoundError(f"Missing config: {config_path}")

with config_path.open("r", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)

config["project"]["output_dir"] = OUTPUT_DIR
config["models"]["use_real_model"] = bool(USE_REAL_MODEL)

# Use real MedMCQA when USE_REAL_MODEL=True.
config["dataset"]["use_mock_data"] = not bool(USE_REAL_MODEL)

# Generation settings.
config["generation"]["use_mock_generation"] = bool(USE_MOCK_GENERATION)
config["generation"]["provider"] = "mock" if USE_MOCK_GENERATION else GENERATION_PROVIDER

# Local HF generation model. L4 can usually handle Qwen2.5-7B for inference; reduce budget/model if OOM.
if not USE_MOCK_GENERATION and GENERATION_PROVIDER == "local_hf":
    config["models"]["generation_model"] = GENERATION_MODEL
elif not USE_MOCK_GENERATION and GENERATION_PROVIDER == "openai":
    config["models"]["generation_model"] = GENERATION_API_MODEL
    config["generation"]["api_model"] = GENERATION_API_MODEL
    config["generation"]["api_batch_size"] = int(GENERATION_API_BATCH_SIZE)
    config["generation"]["api_max_output_tokens"] = 2400
    config["generation"]["api_key_env"] = OPENAI_API_KEY_ENV

# LLM planning agent.
config["planning"]["strategy"] = PLANNING_STRATEGY
config["planning"]["agent_provider"] = LLM_AGENT_PROVIDER
config["planning"]["agent_model"] = LLM_AGENT_MODEL
config["planning"]["api_key_env"] = OPENAI_API_KEY_ENV
config["planning"]["abort_on_error"] = True
if PLANNING_STRATEGY == "llm_agent":
    config["mixture"]["strategy"] = "agent_guided"

# Optional LLM medical critic. This scaling run disables it by default.
if USE_MEDICAL_CRITIC and MEDICAL_CRITIC_PROVIDER == "openai" and not os.environ.get(MEDICAL_CRITIC_API_KEY_ENV):
    try:
        from google.colab import userdata

        api_key = userdata.get(MEDICAL_CRITIC_API_KEY_ENV)
        if api_key:
            os.environ[MEDICAL_CRITIC_API_KEY_ENV] = "".join(api_key.split())
    except Exception:
        pass
if USE_MEDICAL_CRITIC and MEDICAL_CRITIC_PROVIDER == "openai" and not os.environ.get(MEDICAL_CRITIC_API_KEY_ENV):
    raise RuntimeError(
        f"Set {MEDICAL_CRITIC_API_KEY_ENV} in Colab Secrets or os.environ before using the OpenAI medical critic."
    )

config["medical_critic"] = {
    "enabled": bool(USE_MEDICAL_CRITIC),
    "provider": MEDICAL_CRITIC_PROVIDER,
    "model": MEDICAL_CRITIC_MODEL,
    "local_model": MEDICAL_CRITIC_LOCAL_MODEL,
    "api_key_env": MEDICAL_CRITIC_API_KEY_ENV,
    "fail_closed": bool(MEDICAL_CRITIC_FAIL_CLOSED),
    "abort_on_error": True,
    "preflight_check": bool(USE_MEDICAL_CRITIC and MEDICAL_CRITIC_PROVIDER == "openai"),
    "max_new_tokens": 384,
}

# Training settings.
config["training"]["enabled"] = bool(USE_REAL_TRAINING)
config["training"]["dry_run"] = not bool(USE_REAL_TRAINING)

if EVAL_SAMPLES_PER_DOMAIN:
    config["dataset"]["eval_samples_per_domain"] = int(EVAL_SAMPLES_PER_DOMAIN)

if TRAIN_POOL_SAMPLES_PER_DOMAIN:
    config["dataset"]["train_pool_samples_per_domain"] = int(TRAIN_POOL_SAMPLES_PER_DOMAIN)

if TOTAL_SYNTHETIC_BUDGET:
    config["generation"]["total_budget"] = int(TOTAL_SYNTHETIC_BUDGET)

if MIN_SAMPLES_PER_DOMAIN:
    config["planning"]["min_samples_per_domain"] = int(MIN_SAMPLES_PER_DOMAIN)

if MAX_TRAINING_STEPS:
    config["training"]["max_steps"] = int(MAX_TRAINING_STEPS)

runtime_config_path = Path("/content/autodata_runtime_config.yaml")
save_config(config, runtime_config_path)

print("Runtime config:", runtime_config_path)
print(yaml.safe_dump(config, sort_keys=False, allow_unicode=True))


## 7. Optional Smoke Tests

This verifies the package before running the pipeline. It is quick in smoke mode.

In [ ]:
RUN_TESTS = True

if RUN_TESTS:
    subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)

## 8. Run AutoData

Single run executes the full loop once. Strategy comparison runs one loop for each listed strategy and writes a comparison report.

In [ ]:
import os, urllib.request, urllib.error

uses_openai = (GENERATION_PROVIDER == "openai") or (PLANNING_STRATEGY == "llm_agent" and LLM_AGENT_PROVIDER == "openai") or (USE_MEDICAL_CRITIC and MEDICAL_CRITIC_PROVIDER == "openai")
if uses_openai:
    key = os.environ.get(OPENAI_API_KEY_ENV, "")
    print("key present:", bool(key), "length:", len(key or ""))
    req = urllib.request.Request(
        "https://api.openai.com/v1/models",
        headers={"Authorization": f"Bearer {key}"},
    )
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            print("OpenAI reachable:", resp.status)
    except urllib.error.HTTPError as e:
        print("HTTP status:", e.code)
        print(e.read().decode("utf-8")[:1000])
        raise
    except Exception as e:
        print(type(e).__name__, str(e))
        raise
else:
    print("OpenAI preflight skipped.")


In [ ]:
from copy import deepcopy
import json
from pathlib import Path

if RUN_SYNTHETIC_SCALING:
    from autodata.loop.run_loop import run_autodata_loop

    scaling_dir = Path(OUTPUT_DIR) / "synthetic_scaling"
    scaling_dir.mkdir(parents=True, exist_ok=True)
    scaling_rows = []
    for scale_index, budget in enumerate(SYNTHETIC_SCALE_BUDGETS):
        scale_config = deepcopy(config)
        scale_config["generation"]["total_budget"] = int(budget)
        scale_config["project"]["output_dir"] = str(scaling_dir)
        scale_config["project"]["seed"] = int(config.get("project", {}).get("seed", 42)) + scale_index
        print("=" * 80)
        print(f"Running GPT synthetic scaling budget={budget}")
        result = run_autodata_loop(scale_config)
        run_dir = Path(result.run_dir)
        summary = json.loads((run_dir / "round_summary.json").read_text(encoding="utf-8"))
        row = {
            "budget": int(budget),
            "run_dir": str(run_dir),
            "base_overall_accuracy": summary["base_overall_accuracy"],
            "after_overall_accuracy": summary["after_overall_accuracy"],
            "overall_gain": summary["overall_gain"],
            "generated_count": summary["generated_count"],
            "accepted_count": summary["accepted_count"],
            "mixture_sample_count": summary["mixture_sample_count"],
            "training_status": summary["training_status"],
        }
        scaling_rows.append(row)
        print(json.dumps(row, indent=2))

    results_path = scaling_dir / "synthetic_scaling_results.json"
    results_path.write_text(json.dumps(scaling_rows, indent=2), encoding="utf-8")
    run_output = {"scaling_dir": str(scaling_dir), "scaling_rows": scaling_rows, "results_path": str(results_path)}
    print("Scaling results:", results_path)
elif RUN_STRATEGY_COMPARISON:
    from autodata.experiments.strategy_comparison import run_strategy_comparison

    strategy_list = [item.strip() for item in STRATEGIES.split(',') if item.strip()]
    run_output = run_strategy_comparison(config, strategies=strategy_list)
    print('Comparison directory:', run_output['comparison_dir'])
else:
    from autodata.loop.run_loop import run_autodata_loop

    result = run_autodata_loop(config)
    run_output = {'run_dir': result.run_dir}
    print('Run directory:', result.run_dir)
    print('Base accuracy:', f'{result.evaluation_base.overall_accuracy:.3f}')
    print('After accuracy:', f'{result.evaluation_after.overall_accuracy:.3f}')
    print('Training status:', result.training_report.status)


## 9. Inspect Results

The important files are `round_summary.json`, `data_plan.json`, `verification_report.json`, `mixture_report.json`, and `next_round_recommendation.json`.

In [ ]:
import json
from pathlib import Path

try:
    import pandas as pd
except ImportError:
    pd = None

if RUN_SYNTHETIC_SCALING:
    rows = run_output["scaling_rows"]
    print("Scaling results:")
    print(json.dumps(rows, indent=2))
    if pd:
        display(pd.DataFrame(rows))

    import matplotlib.pyplot as plt

    budgets = [0] + [row["budget"] for row in rows]
    base_accuracy = rows[0]["base_overall_accuracy"] if rows else 0.0
    synthetic_accuracy = [base_accuracy] + [row["after_overall_accuracy"] for row in rows]
    plt.figure(figsize=(8, 5.5))
    plt.plot(budgets, [value * 100 for value in synthetic_accuracy], marker="s", color="#ff7f0e", label="Synthetic GPT + LLM agent")
    plt.axhline(base_accuracy * 100, color="gray", linestyle="--", linewidth=1, label=f"Base model ({base_accuracy * 100:.1f}%)")
    for x, y in zip(budgets, synthetic_accuracy):
        plt.text(x, y * 100 + 0.15, f"{y * 100:.1f}", ha="center", fontsize=8, color="#ff7f0e")
    plt.title("Qwen2.5-1.5B on MedMCQA: GPT Synthetic Scaling")
    plt.xlabel("Fine-tuning Samples")
    plt.ylabel("Accuracy (%)")
    plt.xticks(budgets)
    plt.grid(True, linestyle="--", alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plot_path = Path(run_output["scaling_dir"]) / "synthetic_scaling_curve.png"
    plt.savefig(plot_path, dpi=160)
    print("Saved plot:", plot_path)
    plt.show()
elif RUN_STRATEGY_COMPARISON:
    comparison_path = Path(run_output['comparison_dir']) / 'strategy_comparison.json'
    comparison = json.loads(comparison_path.read_text(encoding='utf-8'))
    rows = comparison['strategies']
    if pd:
        display(pd.DataFrame(rows)[[
            'strategy',
            'base_overall_accuracy',
            'after_overall_accuracy',
            'overall_gain',
            'weakest_domain_improvement',
            'strong_domain_drop',
            'accepted_ratio',
            'training_status',
        ]])
    else:
        print(json.dumps(rows, indent=2))
else:
    run_dir = Path(run_output['run_dir'])
    summary = json.loads((run_dir / 'round_summary.json').read_text(encoding='utf-8'))
    data_plan = json.loads((run_dir / 'data_plan.json').read_text(encoding='utf-8'))
    verification = json.loads((run_dir / 'verification_report.json').read_text(encoding='utf-8'))
    mixture = json.loads((run_dir / 'mixture_report.json').read_text(encoding='utf-8'))
    next_round = json.loads((run_dir / 'next_round_recommendation.json').read_text(encoding='utf-8'))
    medical_critic_report = None
    medical_critic_path = run_dir / 'medical_critic_report.json'
    if medical_critic_path.exists():
        medical_critic_report = json.loads(medical_critic_path.read_text(encoding='utf-8'))

    print('Run dir:', run_dir)
    print('Summary:')
    print(json.dumps({
        'base_overall_accuracy': summary['base_overall_accuracy'],
        'after_overall_accuracy': summary['after_overall_accuracy'],
        'overall_gain': summary['overall_gain'],
        'generated_count': summary['generated_count'],
        'accepted_count': summary['accepted_count'],
        'mixture_sample_count': summary['mixture_sample_count'],
        'training_status': summary['training_status'],
        'next_round_focus_domains': summary['next_round_focus_domains'],
    }, indent=2))
    if medical_critic_report:
        print('Medical critic:')
        print(json.dumps({
            'enabled': medical_critic_report.get('enabled'),
            'provider': medical_critic_report.get('provider'),
            'model': medical_critic_report.get('model'),
            'input_count': medical_critic_report.get('input_count'),
            'accepted_count': medical_critic_report.get('accepted_count'),
            'rejected_count': medical_critic_report.get('rejected_count'),
            'rejection_reasons': medical_critic_report.get('rejection_reasons', {}),
        }, indent=2))
    else:
        print('Medical critic disabled or report not found.')

    if pd:
        plan_rows = []
        for domain, item in data_plan['plan'].items():
            plan_rows.append({
                'domain': domain,
                'num_samples': item['num_samples'],
                'data_type': item['data_type'],
                'reason': item['reason'],
                'generation_guidance': item.get('generation_guidance', ''),
                'accepted': verification['accepted_by_domain'].get(domain, 0),
                'mixture': mixture['domain_distribution'].get(domain, 0),
                'gain': next_round['per_domain_gain'].get(domain, 0.0),
                'efficiency': next_round['learning_efficiency_by_domain'].get(domain, 0.0),
            })
        display(pd.DataFrame(plan_rows))
    else:
        print('Data plan:', json.dumps(data_plan, indent=2))


## 10. Peek At Generated Samples

Use this to sanity-check the generated SFT format before training.

In [ ]:
if RUN_SYNTHETIC_SCALING:
    for row in run_output["scaling_rows"]:
        run_dir = Path(row["run_dir"])
        sample_path = run_dir / "verified_samples.jsonl"
        print("=" * 80)
        print("Budget:", row["budget"])
        print("Sample file:", sample_path)
        with sample_path.open('r', encoding='utf-8') as handle:
            for idx, line in enumerate(handle):
                print(json.dumps(json.loads(line), indent=2)[:1200])
                if idx >= 1:
                    break
elif not RUN_STRATEGY_COMPARISON:
    run_dir = Path(run_output['run_dir'])
    sample_path = run_dir / 'verified_samples.jsonl'
    print('Sample file:', sample_path)
    with sample_path.open('r', encoding='utf-8') as handle:
        for idx, line in enumerate(handle):
            print(json.dumps(json.loads(line), indent=2)[:1200])
            if idx >= 2:
                break
else:
    print('Open each run_dir from the comparison table to inspect its verified_samples.jsonl file.')


## 11. Zip The Latest Run

This is optional. It creates a zip archive beside the run directory for download or sharing.

In [ ]:
from shutil import make_archive

if RUN_SYNTHETIC_SCALING:
    scaling_dir = Path(run_output["scaling_dir"])
    archive_path = make_archive(str(scaling_dir), 'zip', root_dir=scaling_dir)
    print('Created:', archive_path)
elif not RUN_STRATEGY_COMPARISON:
    run_dir = Path(run_output['run_dir'])
    archive_path = make_archive(str(run_dir), 'zip', root_dir=run_dir)
    print('Created:', archive_path)
else:
    comparison_dir = Path(run_output['comparison_dir'])
    archive_path = make_archive(str(comparison_dir), 'zip', root_dir=comparison_dir)
    print('Created:', archive_path)
